# 03 — Entraîner et mesurer

**Notebook de cours — `structured-latent-dynamics`**

Le notebook 00 a posé la question du dépôt et le piège du rollout libre. Le notebook 01 a
construit le bloc $2\times2$. Le notebook 02 a construit le wrapper complet
$\psi/\bar A/\pi$ autour de ce bloc, prouvé sa dégénérescence de jauge, et rendu la
comparaison entre bras équitable à $\pm5\%$ de budget. Les trois notebooks précédents
construisent un objet — ce chapitre l'entraîne, et surtout, distingue deux choses que la
suite du dépôt ne confond jamais : ce que l'optimiseur **reçoit** à chaque pas (la loss),
et ce que le lecteur humain **lit** pour juger si le modèle a appris quelque chose (une
poignée de métriques calculées bien après coup, sur un rollout que la loss n'a jamais vu
de cette façon).

Cette distinction n'est pas un détail de comptabilité. Un point central de ce dépôt —
`PROTOCOL.md` §27.8 le mesure noir sur blanc plus loin dans ce notebook — est qu'une loss
composite peut décroître sur toute la durée d'un entraînement de 8000 mises à jour, sans
divergence, pendant que le modèle échoue la porte de contrôle positif qui juge s'il a
appris quoi que ce soit d'utile. Ce chapitre construit les deux objets — la loss et la
lecture — dans cet ordre, pour que le lecteur voie où et pourquoi ils peuvent diverger.

Ce notebook fait tourner un entraînement, mais un seul, minuscule et explicitement marqué
comme tel (§6) : ce qu'il prouve ailleurs, il le prouve par lecture du code réel de `sld`
et par les artefacts déjà écrits sur disque par le dépôt.

## Plan et conventions

| § | Contenu |
|---|---|
| 1 | Ce qu'on optimise n'est pas ce qu'on lit — la tension qui porte le chapitre |
| 2 | L'objectif JEPA : les deux encodeurs, la cible EMA, le stop-gradient |
| 3 | Les quatre termes de la loss, lus sur un vrai `metrics.csv` du dépôt |
| 4 | Les métriques de lecture : $e_H$, VPT, les baselines arm-neutres, la porte de contrôle |
| 5 | Les gardes : le clip de gradient, la divergence typée par artefact |
| 6 | Démonstration vivante : une mini-boucle d'entraînement, explicitement une maquette |
| 7 | Le piège du chapitre, conclusion, et le pont vers le notebook 04 |

**Convention de citation**, reprise sans changement de la série (notebook 02, §0) : un
chiffre **[dépôt]** vient d'un fichier nommé du dépôt et n'est jamais recalculé à la
volée ; un chiffre **[reproduit ici]** est recalculé par une cellule de code qui suit, à
chaque exécution, à partir d'artefacts réels ; un chiffre **[maquette]** vient de la
mini-boucle jouet du §6 — il n'est **ni** mesuré **ni** dépôt, et ce notebook ne l'étiquette
jamais autrement.

**Sources citées dans ce notebook** : `src/sld/objectives/jepa.py`, `src/sld/objectives/
vicreg.py`, `src/sld/training/trainer.py`, `src/sld/experiments/protocol.py` ;
`PROTOCOL.md` §11.7, §12.1-§12.4, §14.9-§14.10, §15.1-§15.3, §17.1, §27.1, §27.7-§27.9 ;
`REGLES.md` §3, §4, §5 ; `tests/objectives/test_jepa.py` ; le résultat du facteur primaire
`results_primary_factorial/` (racine gitignorée — `FINDINGS.md` §11) et son sidecar
versionné `reports/primary/state_scale.json`.

In [ ]:
from pathlib import Path
import csv
import json
import tempfile
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn

np.set_printoptions(precision=6, suppress=False, linewidth=110)
torch.set_printoptions(precision=6, sci_mode=False)
plt.rcParams.update({
    "figure.figsize": (10.5, 3.8),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "legend.fontsize": 8.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
})

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

print("numpy", np.__version__, "| torch", torch.__version__)
print("REPO =", REPO)

## 1. Ce qu'on optimise n'est pas ce qu'on lit

Le notebook 00 (§1) a établi la récurrence de l'erreur d'un predictor entraîné en un pas
et lâché en rollout libre : $e_{t+1} = J_t e_t + \varepsilon_t$, et a montré qu'une
erreur à un pas excellente peut composer, pas après pas, en une divergence de rollout.
Ce chapitre en tire la conséquence pratique pour l'entraînement lui-même : **la quantité
que l'optimiseur reçoit à chaque mise à jour et la quantité que le protocole lit pour
juger un modèle ne sont pas la même quantité, ni calculée sur les mêmes données.**

Ce que l'optimiseur reçoit (§2-§3) : une combinaison pondérée de termes calculés sur un
batch d'épisodes d'entraînement, en partie en mode enseigné (le predictor voit la vraie
trajectoire à chaque pas), en partie en rollout libre court, plus deux régularisateurs
anti-collapse qui ne mesurent rien de la qualité du rollout.

Ce que le protocole lit (§4) : un rollout libre complet de 25 pas, sur le split de
validation ou de test, décodé dans les coordonnées de l'état vrai, comparé à des
baselines qui n'ont rien appris, sous une porte de contrôle positif pré-enregistrée avant
qu'aucun chiffre n'existe.

Rien ne garantit a priori que ces deux quantités bougent ensemble. §7 montre, sur le
facteur primaire du dépôt, un cas où elles ne le font pas.

## 2. L'objectif JEPA : deux encodeurs, une cible EMA, un stop-gradient

`src/sld/objectives/jepa.py` (158 lignes) porte deux objets. Le premier, `EMATarget`,
enveloppe une copie de l'encodeur **online** (celui qui reçoit le gradient) et maintient
une copie **cible**, mise à jour après chaque pas d'optimiseur par une moyenne mobile
exponentielle :

$$\boxed{\;\xi \;\leftarrow\; \tau\,\xi \;+\; (1-\tau)\,\theta\;}$$

où $\theta$ sont les paramètres online et $\xi$ les paramètres cible ($\tau=0.996$ par
défaut dans `EMATarget`, `tau_ema=0.99` dans la configuration du `Trainer`). Le second,
`JEPALoss`, compare la prédiction du predictor à la sortie de cet encodeur cible sur la
**même observation**, jamais à l'encodeur online lui-même — c'est ce qui empêche
l'effondrement trivial où encodeur et predictor convergeraient ensemble vers une
constante en ignorant la dynamique.

Dans `Trainer._composite_loss` (`src/sld/training/trainer.py`), les deux vues sont la
même fenêtre d'observations `x_bt`, passée par les deux encodeurs : `z = encoder(x_bt)`
(online) et `z_tgt = ema(x_bt)` (cible). Le predictor n'avance que `z` (ou sa levée
`h = psi(z)` pour un bras structuré) ; la cible de régression à chaque pas futur est
toujours lue dans `z_tgt`, jamais recalculée par l'online.

**Le double verrou contre le bug silencieux.** `REGLES.md` §5 le nomme explicitement :
*« EMA + stop-gradient : vérifier par test que les paramètres target ne reçoivent aucun
gradient — le bug silencieux classique de JEPA. »* Deux gardes indépendantes le
préviennent ici, et ni l'une ni l'autre n'est censée suffire seule (docstring
`EMATarget`, « belt and suspenders ») : la ceinture est `requires_grad_(False)` posé sur
chaque paramètre cible à la construction (`EMATarget.__init__`) ; les bretelles sont le
`.detach()` que `JEPALoss.forward` applique lui-même à `target_z`, quoi que l'appelant
en ait déjà fait. La cellule suivante ne l'affirme pas — elle le vérifie, sur un
`backward()` réel.

**Ce que compare `JEPALoss`.** Avec $z^{\text{sg}} = \mathrm{sg}[z_{\text{cible}}]$ :

$$\boxed{\begin{aligned}
\text{pred} &= \mathrm{MSE}\big(\pi(\hat h),\; z^{\text{sg}}\big) \\
\text{consistency} &= \mathrm{MSE}\big(\hat h,\; \mathrm{sg}[\psi(z^{\text{sg}})]\big)
  \quad \text{(familles levées seulement)} \\
\text{total} &= \text{pred} \;+\; \lambda_c \cdot \text{consistency}
\end{aligned}}$$

$\pi$ et $\psi$ sont l'identité pour une famille non levée (`mlp`, `transformer`,
`min_ssm`) ; pour une famille levée (`mlp_k`, `transformer_k`, `ssm_k`), `consistency`
compare l'état interne brut $\hat h$ du predictor à la levée **arrêtée en gradient** de
la cible — `psi` ne doit jamais pouvoir tirer la cible vers $\hat h$ par ce terme,
seulement s'entraîner par le chemin de prédiction. Passer `h_hat` sans `lift` (ou
l'inverse) lève une `ValueError` : c'est le bug de câblage à moitié fait que la classe
refuse explicitement.

In [ ]:
from sld.objectives.jepa import EMATarget, JEPALoss

# Démonstration vivante du double verrou -- miroir exact de
# tests/objectives/test_jepa.py::test_ema_gold_gradient_never_leaks_into_target,
# le test que le dépôt possède déjà pour ce mécanisme.
gen = torch.Generator().manual_seed(0)
online = nn.Linear(3, 4)
target = EMATarget(online, tau=0.9)
pred_net = nn.Linear(4, 4)

x = torch.randn(5, 3, generator=gen)
z_online = online(x)
h_hat = pred_net(z_online)
z_target = target(x)

loss = JEPALoss()(h_hat, z_target)["total"]
loss.backward()

for p in target.target.parameters():
    assert p.grad is None, "fuite de gradient dans la cible EMA"

grads = []
for module in (online, pred_net):
    for p in module.parameters():
        assert p.grad is not None
        assert torch.isfinite(p.grad).all()
        grads.append(p.grad)
assert any((g != 0).any() for g in grads)

print("cible EMA : grad is None pour", sum(1 for _ in target.target.parameters()), "paramètres")
print("chemin online + predictor : grad fini et non nul pour",
      sum(1 for _ in online.parameters()) + sum(1 for _ in pred_net.parameters()), "paramètres")

**Lecture.** Le `backward()` a bien traversé le chemin online $\to$ predictor (les deux
`assert` de fin de boucle passent), et n'a laissé **aucune** trace sur les paramètres de
`target.target` : leur `.grad` reste exactement `None`, pas une valeur proche de zéro —
`requires_grad=False` les exclut du graphe avant même que l'accumulation de gradient ne
puisse s'y poser. C'est la même propriété que `tests/objectives/test_jepa.py::
test_ema_gold_gradient_never_leaks_into_target` verrouille dans la suite du dépôt ; cette
cellule ne fait qu'y rejouer les mêmes gestes en direct.

**Pourquoi un terme anti-collapse est nécessaire malgré ce double verrou.** Le
§14.9-§14.10 de `PROTOCOL.md` explique pourquoi ce schéma seul — encodeur online +
cible EMA + stop-gradient, sans contraste négatif — ne suffit pas à empêcher un
effondrement vers une sortie constante : c'est exactement la configuration que l'étude
d'ablation de BYOL rapporte comme s'effondrant, la tête de prédiction y étant ce qui
empêche le collapse. Ce dépôt retient à la place le mécanisme anti-collapse non
contrastif standard — les termes `variance`/`covariance` de VICReg — et c'est le sujet du
§3.

Une précision honnête, pour ne pas confondre deux objets voisins : `PROTOCOL.md` §14.9-
§14.10 décrit l'objectif de **pré-entraînement de la représentation partagée**, propre à
la phase D (deux vues à décalage temporel, aucun predictor dans la boucle). La loss que
ce chapitre entraîne est celle du `Trainer`, utilisée par les six bras du facteur
primaire (§3 et §7) : encodeur et predictor s'y entraînent ensemble (`representation.kind
= identity` pour ce facteur, §27.1), et elle réutilise exactement les deux mêmes briques
— cible EMA à stop-gradient, anti-collapse VICReg — pour la même raison structurelle.

## 3. Les quatre termes de la loss, lus sur un vrai `metrics.csv`

`Trainer._composite_loss` ne s'arrête pas à `JEPALoss`. Chaque mise à jour mélange un
passage **enseigné** (`predictor.step` sur le vrai préfixe, le predictor ne réinjecte
jamais sa propre sortie) et un passage en **rollout libre** court (`predictor.rollout`
sur `h_fr` pas depuis $h_0$), puis ajoute les deux régularisateurs anti-collapse de
VICReg sur l'embedding levé :

$$\boxed{\begin{aligned}
\mathcal{L}_{\text{total}} \;=\;\; & w_{tf}\cdot\text{JEPA}_{\text{enseigné}}
  \;+\; w_{fr}\cdot\text{JEPA}_{\text{libre},\,h_{fr}} \\
  & +\; w_{var}\cdot\underbrace{\tfrac{1}{d}\textstyle\sum_j \max(0,\,\gamma_v - \sigma_j(z))}_{\text{var}}
  \;+\; w_{cov}\cdot\underbrace{\tfrac{1}{d}\textstyle\sum_{i\neq j} C_{ij}(z)^2}_{\text{cov}}
  \;+\; \mu_J \cdot \text{jac\_pen}
\end{aligned}}$$

`jac_pen` (pondéré par `mu_j`) est la pénalité de Jacobienne de la campagne μ_J, hors
sujet ici et à `mu_j = 0.0` sur toutes les cellules du facteur primaire lu plus bas.

`var` et `cov` viennent de `src/sld/objectives/vicreg.py`
([Bardes, Ponce & LeCun (2021), *VICReg: Variance-Invariance-Covariance Regularization
for Self-Supervised Learning*](https://arxiv.org/abs/2105.04906)) : `var` est une charnière
sur l'écart-type par dimension ($\gamma_v=1$ par défaut) — elle **pénalise une dimension
qui s'effondre** — et `cov` pénalise la covariance hors-diagonale entre dimensions —
elle **décorrèle** l'embedding. C'est le mécanisme anti-collapse annoncé au §2 : sans lui,
le double verrou EMA + stop-gradient seul n'empêche pas un effondrement vers une sortie
constante.

Ces quatre termes mesurés — `pred`, `consistency`, `var`, `cov` — **sont exactement**
les quatre colonnes homonymes de `metrics.csv` que `Trainer` écrit à chaque époque
(`METRICS_HEADER`, `src/sld/training/trainer.py`). La cellule suivante les lit depuis
deux vraies cellules du facteur primaire, sans les recalculer.

In [ ]:
ROOT = REPO / "results_primary_factorial" / "primary_factorial"

# Garde d'existence : cette racine est gitignoree (FINDINGS.md §11 -- le contenu
# scientifique vit dans PROTOCOL.md et les sidecars de figure, pas dans les runs
# bruts), donc absente d'un clone frais. Le notebook doit rester exécutable sans elle.
if not ROOT.exists():
    print(f"[garde] {ROOT} absente -- racine gitignoree, absente d'un clone frais "
          "(FINDINGS.md §11). Les lectures de ce §3 s'arrêtent ici ; les valeurs "
          "publiées restent citées en texte plus bas, avec leur source PROTOCOL.md.")
    example_runs = {}
else:
    import yaml

    example_runs = {
        "mlp_k (levé, off/on structuré)": "d12ac614-3cb3e669",
        "mlp (bras plat)": "7b73e3d6-3cb3e669",
    }
    loaded = {}
    for label, run_hash in example_runs.items():
        run_dir = ROOT / run_hash
        cfg = yaml.safe_load((run_dir / "config.yaml").read_text())
        with open(run_dir / "metrics.csv") as f:
            rows = list(csv.DictReader(f))
        loaded[label] = {"cfg": cfg, "rows": rows}
        print(f"{label}: family={cfg['family']} seed={cfg['seed']} lambda_c={cfg['lambda_c']} "
              f"n_epochs={len(rows)} grad_clip={cfg.get('grad_clip')}")
    example_runs = loaded

**Lecture.** `lambda_c = 0.0` pour les deux cellules — c'est la valeur par défaut de
`Trainer`, jamais surchargée sur le facteur primaire (aucune occurrence de `lambda_c`
dans `PROTOCOL.md`). Conséquence directe de la formule du §2 : la colonne `consistency`
est écrite dans `metrics.csv` — c'est un diagnostic réel — mais elle **ne pèse pour rien
dans le gradient** sur cette grille, puisque `total = pred + 0.0 * consistency`. Une
courbe `consistency` qui monte n'est donc pas un signe que l'entraînement se dégrade :
c'est un canal que personne n'optimise, laissé visible.

`mlp` n'est pas une famille levée : `hasattr(predictor, "lift")` est faux, `JEPALoss`
n'y reçoit jamais `h_hat`/`lift`, et `consistency` y est identiquement nulle à chaque
époque — pas petite, **exactement** 0.0, par construction du code et non par
convergence.

In [ ]:
if example_runs:
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharex=True)
    terms = ["pred", "consistency", "var", "cov"]
    for label, payload in example_runs.items():
        rows = payload["rows"]
        epochs = [int(r["epoch"]) for r in rows]
        for ax, term in zip(axes, terms):
            ax.plot(epochs, [float(r[term]) for r in rows], label=label, lw=1.1)
    for ax, term in zip(axes, terms):
        ax.set_yscale("symlog", linthresh=1e-3)
        ax.set_title(term)
        ax.set_xlabel("époque")
    axes[0].set_ylabel("valeur du terme (symlog)")
    axes[-1].legend(fontsize=7, loc="upper right")
    fig.suptitle("results_primary_factorial -- mlp_k vs mlp, seed 0, 8000 époques", y=1.04)
    fig.tight_layout()
    plt.show()
else:
    print("figure sautée -- racine absente, voir la garde ci-dessus")

**Lecture de la figure.** Axes : époque en abscisse (échelle linéaire, 0 à 8000),
valeur du terme en ordonnée (symlog, pour voir aussi bien les valeurs proches de 0 que
les pics de départ). `pred` est le terme que `w_tf=w_fr=1` optimise directement, et les
deux bras y répondent, mais pas au même rythme : `mlp` chute d'un facteur supérieur à 500
(près de trois ordres de grandeur) dès les cent premières époques (8,86 $\to$ 0,016) ;
`mlp_k`, contraint par le
wrapper structuré, met plus longtemps à atteindre la même chute (0,013 à l'époque 1000,
sous 0,005 à partir de 4000). `var` illustre les deux régimes de la charnière : pour
`mlp`, l'écart-type par dimension dépasse $\gamma_v=1$ dès l'initialisation et le terme
est **exactement** 0 à toutes les époques échantillonnées, de la première à la
dernière — rien à corriger, jamais. Pour `mlp_k`, parti loin de 0 (l'embedding levé
initial est peu dispersé), `var` descend près de 0 vers l'époque 100 puis **y revient
par intermittence** plus tard dans l'entraînement (0,037 à l'époque 1000, 0,075 à 4000) :
la charnière n'est pas un point fixe permanent, elle se rallume chaque fois qu'une
dimension repasse sous le seuil (`vicreg.py`, note d'honnêteté du module : le gradient
n'est nul qu'exactement au seuil, pas au-delà). `cov` oscille sans jamais s'annuler,
pour les deux bras : décorréler complètement l'embedding n'est pas un point fixe stable
de l'entraînement. `consistency`, pour `mlp_k`, monte d'abord avec `pred` et `var` sur
les cent premières époques (0,24 $\to$ 0,49 — rien ne la retient, `lambda_c=0`), puis
retombe sous 0,002 par l'époque 8000, un sous-produit de l'optimisation du reste de la
loss plutôt qu'un effet direct ; elle reste rivée à 0 pour `mlp`, pour les raisons
données plus haut.

**Ce que cette figure ne montre pas.** Rien ici ne parle du rollout libre à l'évaluation :
les quatre courbes sont des moyennes sur un batch d'entraînement, à un horizon
d'entraînement (`h_fr`) qui n'est pas encore celui de la lecture. §4 introduit la
métrique qui, elle, est calculée sur cet autre objet.

## 4. Les métriques de lecture

### 4.1 $e_H$ : l'erreur d'état normalisée, en coordonnées vraies

`PROTOCOL.md` §12.1 fixe la métrique de lecture principale (`H = 25` fixé au §12.3 ci-
dessous) :

$$\boxed{\;e_h \;=\; \sqrt{\frac{1}{B\,d}\sum_{b}\sum_{i}
\left(\frac{\hat x_{b,h,i} - x_{b,h,i}}{\sigma_i^{\text{train}}}\right)^{2}}\;}$$

$\hat x$ est le rollout libre, **décodé dans l'espace d'état vrai** par le readout figé ;
$x$ est l'état généré par l'oracle RK4 ; $\sigma_i^{\text{train}}$ est l'écart-type par
coordonnée, mesuré sur le split train et gelé avant toute évaluation. C'est le NRMSE de
[Vlachas et al. (2020), *Backpropagation algorithms and Reservoir Computing in Recurrent
Neural Networks for the forecasting of complex spatiotemporal dynamics*,
Neural Networks 126:191-217](https://arxiv.org/abs/1910.05266) — la moyenne sur les
épisodes se fait ici sous la racine, et $\sigma$ se prend sur le split train plutôt qu'en
temps, mais la normalisation par coordonnée est celle publiée.

**Pourquoi jamais une MSE dans l'espace latent.** Le notebook 02 (§3.2) a démontré une
orbite de dimension $2r$ du wrapper $\psi/\bar A/\pi$ le long de laquelle la sortie
$\hat z_t$ est exactement inchangée : plusieurs paramétrisations $(\psi,\bar A,\pi)$
gauge-équivalentes produisent le même comportement observable mais des états internes
$h$ différents. Une métrique lue directement dans cet espace interne — ou plus
généralement une comparaison entre deux familles dont l'espace latent n'a pas la même
dimension ou la même paramétrisation (plat pour `mlp`, levé $2r$ pour `mlp_k`) — n'a pas
de sens canonique commun. $e_h$ contourne cela en ne comparant jamais deux latents entre
eux : elle décode toujours vers $x$, la seule coordonnée que les six bras du factoriel
partagent.

### 4.2 VPT : l'horizon-jusqu'au-seuil

`PROTOCOL.md` §12.2 :

$$\boxed{\;\mathrm{VPT}_\tau(e) \;=\; \max\{\,h \in \{0,\dots,H\} : e_j < \tau\ \ \forall j\le h\,\}\;}$$

la longueur du plus long préfixe valide, avec $\tau = 0.5$ (la valeur publiée par Vlachas
et al. sur ce même NRMSE), $\mathrm{VPT}=0$ si $e_1\ge\tau$, et **censuré** (statut
`censored`, valeur `inf`) si le seuil n'est jamais franchi sur l'horizon observé —
substituer $H$ à la place fabriquerait une mesure qui n'a pas été prise.

`REGLES.md` §4 est catégorique : *« MSE jamais seule sur un système chaotique : Lorenz se
lit en VPT (unités de Lyapunov) + fidélité statistique long terme. »* Cette règle a été
écrite pour Lorenz-63 (notebook 00), le cas adversarial de la série. Le système
conducteur de ce chapitre, `CoupledOscillators`, est amorti et **non chaotique** : son
exposant de Lyapunov dominant est $\le 0$, et `PROTOCOL.md` §12.2 le dit explicitement —
diviser un VPT par $\Lambda_1$ y serait dénué de sens, le module renvoie un simple compte
de pas. La leçon générale derrière la règle — ne jamais lire une erreur scalaire seule
quand elle peut se comporter de façon non monotone ou diverger vite — motive quand même
d'introduire le VPT ici, même sans la normalisation Lyapunov qui ne s'applique pas à ce
système.

**VPT est censuré par construction sur ce système.** `PROTOCOL.md` §27.7 (daté
2026-08-14) le mesure, sur les cinq racines existantes au moment où la précision a été
écrite : sur **1346 épisodes d'évaluation** de `results_osc_n/`, `results_osc_delta/`,
`results_osc_decomp/`, `results_osc_hfr/` et `results_ssm_delta/`, **1179 sont censurés,
soit 87,6 %**, et seuls **30 des 162 runs scorés** atteignent $e_h \ge 0.5$ dans les 25
pas ; la médiane de $e_{h,\text{gate}}$ sur `CoupledOscillators` à cet horizon est
**0,1592**, un facteur trois sous $\tau$. Sous la règle de censure conservatrice du
§27.3 (toute comparaison dépendant d'une valeur censurée devient `indeterminate`), un
VPT pris comme statistique primaire renverrait `indeterminate` sur la quasi-totalité des
relations testées — pas parce que les modèles seraient à égalité, mais parce que la
métrique ne peut pas les séparer sur ce système. **La statistique primaire du facteur
primaire est donc $e_H$ à $H=25$**, VPT restant publié (Fig. 2 du dépôt) comme mesure
secondaire avec son statut de censure.

In [ ]:
STATE_SCALE_PATH = REPO / "reports" / "primary" / "state_scale.json"

# reports/ n'est PAS gitignore (contrairement a results_*), donc ce fichier existe
# dans un clone frais -- pas de garde d'existence necessaire ici, juste une lecture.
state_scale = json.loads(STATE_SCALE_PATH.read_text())
stratum = state_scale["strata"]["CoupledOscillators__OURegime__3cb3e669"]
print(json.dumps(stratum, indent=2))

**Lecture.** `PROTOCOL.md` §27.10 (daté 2026-08-18) corrige une valeur copiée d'une
strate différente dans une parenthèse antérieure du §27.1 ; ce sidecar versionné
(`reports/primary/state_scale.json`, digest `409a5317…`) est la mesure de référence pour
la strate réellement exécutée par le facteur primaire, et son digest est exactement celui
lu dans `config.yaml` (§3, `representation_digest`). Rien de ce qui suit ne dépend de sa
valeur exacte — la représentation `identity` reconstruit au plancher numérique quelle
que soit la strate — mais c'est le nombre qui, au §4.3, nourrit réellement $\tau_1$.

### 4.3 Les baselines arm-neutres, et la porte de contrôle positif

`PROTOCOL.md` §12.3 définit deux baselines qui n'apprennent rien — la persistance et
l'extrapolation linéaire, dans les mêmes coordonnées, les mêmes épisodes, le même
$\sigma^{\text{train}}$ — et une porte à deux bornes :

$$\boxed{\;G \;=\; \Big[\,e_{H_{\text{gate}}} < \frac{\min(e^{\text{pers}}_{H_{\text{gate}}},\,e^{\text{lin}}_{H_{\text{gate}}})}{\gamma}\,\Big] \;\wedge\;
\Big[\,e_1 < \tau_1\,\Big]\;}$$

avec $H_{\text{gate}}=25$ (commun à tout le dépôt), $\gamma=5$ (§12.3), et $\tau_1$ dans
sa **seconde version** : `PROTOCOL.md` §15.1 a remplacé la règle initiale
$\tau_1 = 3\cdot e_{\text{recon}}$ (qui se resserrait quand l'encodeur s'améliorait — le
mauvais sens pour un plancher) par

$$\tau_1 \;=\; \frac{e_{1}^{\text{pers}}}{\gamma_1}, \qquad \gamma_1 = 2.$$

**Parité obligatoire, pas recopiée.** La cellule suivante importe ces quatre constantes
directement depuis `src/sld/experiments/protocol.py` — le module que le code de
`gate.py` importe réellement — et vérifie qu'elles valent ce que ce paragraphe vient
d'écrire, plutôt que de faire confiance à la prose.

In [ ]:
from sld.experiments.protocol import GAMMA, GAMMA_1, H_GATE, TAU_VPT, tau1_validity, tau_1

assert H_GATE == 25          # PROTOCOL.md §12.3
assert GAMMA == 5.0          # PROTOCOL.md §12.3
assert TAU_VPT == 0.5        # PROTOCOL.md §12.2
assert GAMMA_1 == 2.0        # PROTOCOL.md §15.1

e_1_pers = stratum["e_1_pers"]
e_recon = stratum["e_recon"]
tau_1_value = tau_1(e_1_pers)
validity = tau1_validity(e_recon, e_1_pers)

assert abs(tau_1_value - e_1_pers / GAMMA_1) < 1e-15  # PROTOCOL.md §15.1, formule vérifiée contre le code

print(f"H_gate={H_GATE}  gamma={GAMMA}  gamma_1={GAMMA_1}  tau={TAU_VPT}")
print(f"strate CoupledOscillators/OURegime : e_1_pers={e_1_pers:.6f}  e_recon={e_recon:.3e}")
print(f"tau_1 = e_1_pers / gamma_1 = {tau_1_value:.6f}")
print("validite (§15.1) :", validity)

**Lecture.** Les quatre constantes importées valent exactement ce que le §4.3 vient de
citer — `sld.experiments.protocol` est, par son propre docstring, le seul endroit du
dépôt d'où `gate.py` et tout appelant sont censés les lire, plutôt que de laisser le
littéral dériver entre `PROTOCOL.md` et le code. Sur la strate `CoupledOscillators` /
`OURegime` du facteur primaire, $\tau_1 \approx 0{,}024$ : `validity["attainable"]` est
`True` (le plancher $e_{\text{recon}}$, à $1{,}9\times10^{-8}$, est six ordres de
grandeur sous ce plafond — l'encodeur identité ne peut jamais être le facteur limitant)
et `validity["discriminant"]` est `True` par construction dès que $\gamma_1>1$. La strate
n'est donc jamais publiée « ungated » : la porte discrimine réellement sur ce facteur.

**Les marges, et la règle des cinq seeds.** `PROTOCOL.md` §12.4 nomme `m_low = e_{H_{\text{gate}}}
\cdot \gamma / \min(e^{\text{pers}}, e^{\text{lin}})` et `m_high = e_1/\tau_1` — les deux
marges lisibles sans inférence (`< 1` est un passage) que `sld.experiments.probe`
documente sous les mêmes noms. **Les cinq seeds doivent tous passer individuellement** —
aucune moyenne, aucun réducteur `k`-parmi-5 : une cellule avec un seul seed en échec est
publiée en échec, nommée, jamais moyennée. §7 revient sur ce que cette règle change dans
la lecture du facteur primaire.

## 5. Les gardes

### 5.1 Le clip de gradient, uniforme et hashé

`PROTOCOL.md` §17.1 déclare une garde d'optimiseur unique, ajoutée après que `min_ssm`
ait explosé à toutes les LR testées en round 3 (norme de gradient au premier pas : `2.4e18`
contre `28.2` pour `mlp`, le même harnais, la même donnée) : `grad_clip = 100`, la norme
L2 globale des gradients, écrêtée juste avant chaque pas d'optimiseur
(`torch.nn.utils.clip_grad_norm_`). **Uniforme** sur les six bras, toutes strates,
tous seeds — une garde d'optimiseur est une propriété de budget, pas un réglage par
bras. **Optionnelle mais hashée quand présente** : `grad_clip` absente laisse
`config_hash` identique à avant l'amendement (aucun run existant ne change d'identité) ;
présente, elle entre dans le hash — un run écrêté est une expérience différente. La
valeur elle-même est un jugement, pas une mesure : la plage publiée est 20-1000, et 100
est le standard de la famille Dreamer, plus de 3× au-dessus de la plus grande norme saine
que le dépôt ait mesurée (~28-30) — une garde-no-op pour les bras qui s'entraînaient déjà
bien, qui n'intercepte que les pas pathologiques.

`grad_clip = 100` est tenu fixe sur tout le facteur primaire (`PROTOCOL.md` §27.1) — la
même valeur que `config.yaml` a déjà rapportée au §3. Et la trace de son activité y est
lisible : au §3, la première ligne de `metrics.csv` pour la cellule `mlp` (non
structurée) portait `grad_norm=145.05` — au-dessus de 100. Cette toute première mise à
jour a donc réellement été écrêtée ; les époques suivantes retombent vite sous le seuil.
`grad_norm` est justement la mesure ajoutée par cet amendement (§17.1, dernier point) :
la norme **avant** écrêtage, publiée à chaque époque, pour qu'une garde qui ne mord
jamais ne reste pas un réglage invisible.

### 5.2 La divergence, typée par artefact, jamais par un message

`PROTOCOL.md` §11.7 (entraînement) et §15.3 (évaluation) déclarent deux classes
terminales distinctes. À l'entraînement, une loss non finie **abandonne le run sans
artefact de succès** — ni `ckpt.pt` valide, ni `run_complete.json` — et écrit
atomiquement `failure.json` dans le dossier du run, avec l'identité du run, l'époque, la
raison, les valeurs fautives et la classe d'exception ; l'index classe cette cellule
`divergence`, une classe distincte de `ok` et de `missing`, préservée par une reprise
(jamais réentraînée par erreur). Une rejoue diagnostique est permise dans un espace de
noms séparé ; substituer un autre seed ou une autre LR dans le résultat primaire ne l'est
pas. À l'évaluation — un checkpoint qui a fini son entraînement proprement mais dont le
rollout de scoring quitte lui-même le domaine fini — `PROTOCOL.md` §15.3 ajoute une
seconde classe terminale, `divergence (evaluation)`, avec son propre artefact
`eval_failure.json` ; contrairement à un réentraînement, une réévaluation y est
déterministe par construction (l'évaluation est une fonction pure d'un checkpoint figé),
donc vérifiable, pas seulement promise.

**Pourquoi un artefact et jamais un message.** Un dossier de run partiel sans sentinelle
de complétion est, sur disque, indiscernable d'une cellule qui n'a jamais tourné — une
reprise réentraînerait silencieusement une cellule déjà tranchée. Un message de log n'est
lu que par qui regarde au bon moment ; un artefact JSON est une donnée que l'index, le
validateur du factoriel et une reprise future peuvent tous interroger, longtemps après
que le run a fini.

La cellule suivante ne cite pas cette règle, elle la déclenche : une configuration dont le
seul défaut est une LR absurde, sur les mêmes système et wrapper que §6, pour voir
l'artefact réel plutôt qu'une prose sur l'artefact.

In [ ]:
from sld.data.generate import build_dataset
from sld.data.regimes import OURegime
from sld.systems.coupled_oscillators import CoupledOscillators
from sld.training.trainer import Trainer

# MAQUETTE -- LR volontairement absurde pour déclencher §11.7, rien de mesuré ici.
toy_data = build_dataset(
    system=CoupledOscillators(), regime=OURegime(), obs=None,
    n_episodes=16, T=30, dt=0.05, seed=0,
)
divergent_cfg = {
    "exp_name": "nb03_divergence_demo",
    "seed": 0,
    "family": "mlp_k",
    "d_x": 4, "d_z": 4, "d_u": 1,
    "delta": 0.05, "radius": "bounded", "r": 2,
    "target_params": 2000,
    "epochs": 50,
    "batch_episodes": 8,
    "lr": 50.0,
    "h_fr": 5,
    "results_root": tempfile.mkdtemp(prefix="sld_nb03_divergence_"),
}
divergent_trainer = Trainer(divergent_cfg)
try:
    divergent_trainer.fit(toy_data)
    raise AssertionError("attendu : la loss diverge sous lr=50.0")
except RuntimeError as exc:
    print("run avorté comme attendu :", exc)

failure_path = divergent_trainer.run_dir / "failure.json"
assert failure_path.exists()
print(json.dumps(json.loads(failure_path.read_text()), indent=2))

**Lecture.** `RuntimeError` interrompt `fit`, et `failure.json` porte exactement les
champs que §5.2 vient de nommer : `config_hash`, `data_hash`, l'époque (`0` — la toute
première mise à jour), la raison (`"non-finite loss"`), les champs fautifs
(`val_loss` non fini) et la classe d'exception. Aucun `ckpt.pt` ni `run_complete.json`
n'existe dans ce dossier — c'est la propriété qui rend la reprise sûre : un futur appel
sur cette identité verrait l'artefact `divergence`, jamais un dossier ambigu.

## 6. Démonstration vivante : une mini-boucle d'entraînement

Cette section fait tourner le vrai `Trainer` du dépôt, en configuration minimale, pour
qu'une `history` complète existe sous les yeux du lecteur — **pas pour produire un
résultat**. Tous les chiffres qui en sortent sont des **[maquette]** : ni mesurés, ni
dépôt, jamais recités comme tels ailleurs dans ce notebook. Le système conducteur reste
`CoupledOscillators` sous `OURegime`, le même que celui du facteur primaire lu au §3 et
au §7 — mais à une échelle jouet : $d_z=4$, 16 épisodes de 30 pas à $dt=0{,}05$,
`family="mlp_k"` (le bras structuré, pour retrouver les quatre termes du §3 dans une
seule `history`), 50 mises à jour (`epochs=50` — le `Trainer` v0 tire un batch par époque,
sans dataloader, donc une époque **est** une mise à jour ici), seed fixe.

In [ ]:
demo_data = build_dataset(
    system=CoupledOscillators(), regime=OURegime(), obs=None,
    n_episodes=16, T=30, dt=0.05, seed=0,
)
demo_root = tempfile.mkdtemp(prefix="sld_nb03_demo_")
demo_cfg = {
    "exp_name": "nb03_demo",
    "seed": 0,
    "family": "mlp_k",
    "d_x": 4, "d_z": 4, "d_u": 1,
    "delta": 0.05, "radius": "bounded", "r": 2,
    "target_params": 2000,
    "epochs": 50,
    "batch_episodes": 8,
    "lr": 1e-2,
    "h_fr": 5,
    "grad_clip": 100,
    "results_root": demo_root,
}

t0 = time.time()
demo_result = Trainer(demo_cfg).fit(demo_data)
elapsed = time.time() - t0
print(f"50 mises à jour en {elapsed:.2f} s")
print("history couvre", len(demo_result["history"]["epoch"]), "époques")
print("métriques finales [maquette] :", demo_result["metrics"])

with open(demo_result["paths"]["metrics_csv"]) as f:
    demo_rows = list(csv.DictReader(f))

**Lecture.** `history` (le dict retourné par `fit`) ne porte que `epoch`/`train_loss`/
`val_loss` plus des clés d'audit à `None` (elles arrivent avec l'audit hors ligne de la
vague 4, jamais calculées ici) — les quatre termes `pred`/`consistency`/`var`/`cov` vivent
dans `metrics.csv`, à `paths["metrics_csv"]`, exactement comme au §3. C'est la même
mécanique de lecture, appliquée à un entraînement de 50 mises à jour au lieu de 8000.
Aucun de ces nombres n'est mesuré ni dépôt : la seed, la taille du jeu et le budget sont
choisis pour qu'une figure existe en quelques secondes, pas pour représenter quoi que ce
soit du facteur primaire.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), sharex=True)
terms = ["pred", "consistency", "var", "cov"]
epochs = [int(r["epoch"]) for r in demo_rows]
for ax, term in zip(axes, terms):
    ax.plot(epochs, [float(r[term]) for r in demo_rows], color="tab:orange", lw=1.3)
    ax.set_title(term)
    ax.set_xlabel("mise à jour")
axes[0].set_ylabel("valeur du terme")
fig.suptitle("[maquette] mini-boucle jouet, 50 mises à jour -- ni mesuré, ni dépôt", y=1.04)
fig.tight_layout()
plt.show()

**Lecture [maquette].** `pred` décroît sur les 50 mises à jour (l'optimiseur le voit
directement) ; `var` décroît sans atteindre 0 dans ce budget minuscule (la charnière
n'est pas encore saturée) ; `cov` reste petit et bruité ; `consistency`, elle, **monte** —
`lambda_c=0.0` par défaut ici aussi, donc rien ne la retient, exactement la lecture faite
au §3 sur les vraies cellules du facteur primaire, à une échelle 160 fois plus courte.
La forme qualitative se reconnaît (`pred` qui chute vite, `consistency` non retenue) ; les
valeurs, elles, ne se comparent à rien — 50 mises à jour sur 16 épisodes jouets n'ont
aucun rapport d'échelle avec 8000 mises à jour sur la donnée réelle du §3.

## 7. Le piège du chapitre, et ce que le facteur primaire en a fait

**Le piège**, maintenant qu'il peut se dire avec les objets du chapitre : une loss
composite qui décroît proprement — `pred` en chute libre, `var`/`cov` sagement bornés,
aucune divergence — ne garantit rien sur $e_H$. Le dépôt le savait en partie : `PROTOCOL.md`
§15.2 a spécifiquement relevé `h_fr` d'entraînement de 5 à 25 pour l'aligner sur
$H_{\text{gate}}$, précisément parce qu'entraîner en rollout libre sur 5 pas pendant que
la porte en score 25 laissait le comportement du modèle au-delà du pas 5 largement non
contraint (le §15.2 le mesure : le rayon spectral local médian d'un bras `mlp_k` passe de
`~1,23` à `h_fr=5` à `~1,05` à `h_fr=25`, sur les mêmes checkpoints). Mais égaliser les
deux horizons ne fait pas disparaître le reste de l'écart : le terme enseigné (`w_tf=1`)
domine toujours la loss d'entraînement, et `var`/`cov` ne mesurent rien de la qualité
d'un rollout — ils empêchent seulement l'embedding de s'effondrer.

**Le facteur primaire en offre une instance concrète, chiffrée.** `PROTOCOL.md` §27.8 :
**30/30 cellules complètes, zéro divergence** — chaque cellule a fini son entraînement de
8000 mises à jour proprement, exactement le régime que ce chapitre vient d'outiller.
Et pourtant, sous la porte de contrôle positif du §4.3 (les cinq seeds doivent tous
passer), seuls `mlp` et `transformer` sont vérifiés `learns` ; les quatre autres bras
échouent — `ssm_k` rate même la borne arm-neutre inférieure sur **5 seeds sur 5** (il ne
bat pas les baselines triviales), quand `mlp_k` la passe sur 4 seeds sur 5 et échoue
seulement sur la borne supérieure $\tau_1$ (`PROTOCOL.md` §27.8.2). Une loss qui
décroît sans jamais diverger et un modèle qui échoue la lecture ne sont pas
contradictoires : ce chapitre vient de construire les deux objets qui expliquent
pourquoi.

Sur ce même facteur, le contraste apparié on/off de $e_H$ (celui que §27.2 certifie sans
confusion, puisque on et off d'un même bras partagent seed, taux et budget) est positif
sur `mlp` et sur `transformer` — porté par les cinq seeds de chacun — c'est-à-dire que la
structure y **coûte**, pas qu'elle paie. Sur le troisième bras, `ssm`, ce même contraste
n'est positif que sur 3 seeds sur 5 et sa moyenne est portée par un seul seed extrême :
`PROTOCOL.md` §27.9 est explicite, ce bras n'est **pas compté comme une troisième
confirmation indépendante**, pour une raison pré-déclarée avant même que la cellule ne
tourne — §27.6 note que la complétion 10/10 du bras `ssm` n'a été mesurée qu'à `n=256`
(§24.0), jamais à `n=32`, le régime de ce facteur, et publie le risque par avance ; §27.8.2
mesure ensuite l'issue : `ssm_k` rate la borne arm-neutre inférieure sur 5 seeds sur 5,
`min_ssm` sur 3 seeds sur 5. Ce que ce facteur établit, dans les mots mêmes du dépôt, est
que **le négatif généralise à deux backbones**, `mlp` et `transformer`. Le chiffrage
complet — les neuf relations, les lectures exploratoires, ce que chacune couvre et où elle
s'arrête — appartient au notebook 04.

### Ce qui a été prouvé ici

Le double verrou EMA + stop-gradient de `JEPALoss`/`EMATarget` a été vérifié en direct
sur un `backward()` réel, en miroir du test que le dépôt possède déjà. Les quatre termes
de la loss composite du `Trainer` — `pred`, `consistency`, `var`, `cov` — ont été lus sur
`metrics.csv` de deux vraies cellules du facteur primaire, avec la nuance que
`consistency` n'y pèse jamais dans le gradient (`lambda_c=0.0`). $e_H$, le VPT et sa
censure par construction sur ce système, les baselines arm-neutres et la porte de
contrôle positif ont été définis depuis `PROTOCOL.md`, avec `H_gate`/`gamma`/`gamma_1`/
`tau_1` vérifiés en parité contre `src/sld/experiments/protocol.py` plutôt que recopiés.
Le clip de gradient et les deux classes de divergence typées par artefact ont été
expliqués et, pour la divergence à l'entraînement, déclenchés en direct avec un vrai
`failure.json`. Une mini-boucle d'entraînement, explicitement une maquette, a montré une
`history` complète en quelques secondes.

**Lien avec `sld`.** `src/sld/objectives/jepa.py` (`EMATarget`, `JEPALoss`),
`src/sld/objectives/vicreg.py` (`variance_loss`, `covariance_loss`),
`src/sld/training/trainer.py` (`Trainer.fit`, `_composite_loss`, `METRICS_HEADER`,
`FAILURE_FILENAME`), `src/sld/experiments/protocol.py` (`H_GATE`, `GAMMA`, `GAMMA_1`,
`tau_1`, `tau1_validity`), `tests/objectives/test_jepa.py`.

**La limite honnête.** Ce notebook prouve le mécanisme de la loss et définit les
métriques de lecture ; il ne prouve rien sur ce que la grille complète, une fois tous ses
bras comparés en détail, établit ou n'établit pas. Il cite le verdict du facteur primaire
comme instance du piège qu'il vient de nommer, pas comme son analyse — celle-ci, les neuf
relations pré-enregistrées et ce que le chemin des campagnes qui y mène a coûté,
appartiennent au notebook suivant.

**Le pont.** La loss est câblée, la lecture est définie, les gardes protègent la boucle.
Reste à comprendre ce que la grille, prise dans son ensemble, a effectivement dit — le
notebook 04.